In [2]:
# !pip install pandas numpy matplotlib seaborn plotly wordcloud nltk textblob scikit-learn transformers streamlit

# !pip install --upgrade nbformat ipykernel plotly

### Data Loading & Inspection

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

Exception ignored in PyObject_HasAttr(); consider using PyObject_HasAttrWithError(), PyObject_GetOptionalAttr() or PyObject_GetAttr():
Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 491, in _call_with_frames_removed
ImportError: PyCapsule_Import could not import module "pandas"


In [4]:
df = pd.read_csv("data/raw/Paypal_Android_iOS_Reviews_2026.csv", encoding="latin-1")

print("Dataset Shape:", df.shape)

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/Paypal_Android_iOS_Reviews_2026.csv'

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 28848 entries, 0 to 28847
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   reviewId              28848 non-null  str    
 1   score                 28848 non-null  int64  
 2   content               28846 non-null  str    
 3   at                    28848 non-null  str    
 4   android_or_ios        28848 non-null  str    
 5   thumbsUpCount         26925 non-null  float64
 6   reviewCreatedVersion  22495 non-null  str    
 7   replyContent          95 non-null     str    
dtypes: float64(1), int64(1), str(6)
memory usage: 6.5 MB


In [5]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
reviewId,28848,28848,8938a0da-4273-4019-865c-16ebf97e251a,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
score,28848.0,NaN,NaN,NaN,3.162923,1.863772,1.0,1.0,4.0,5.0,5.0
content,28846,24446,good,590,NaN,NaN,NaN,NaN,NaN,NaN,NaN
at,28848,27832,5/9/2026 4:48,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
android_or_ios,28848,2,android,26925,NaN,NaN,NaN,NaN,NaN,NaN,NaN
thumbsUpCount,26925.0,NaN,NaN,NaN,1.403268,17.826217,0.0,0.0,0.0,1.0,1290.0
reviewCreatedVersion,22495,191,8.107.0,4544,NaN,NaN,NaN,NaN,NaN,NaN,NaN
replyContent,95,70,"Hi- We're sorry to hear about your recent experience, but we'd love to see if there's something we can do to turn your situation around. Please co...",11,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df.isnull().sum()

reviewId                    0
score                       0
content                     2
at                          0
android_or_ios              0
thumbsUpCount            1923
reviewCreatedVersion     6353
replyContent            28753
dtype: int64

In [7]:
print("Duplicate review IDs:", df["reviewId"].duplicated().sum())

print("\nPlatform distribution:")
print(df["android_or_ios"].value_counts())

print("\nRating distribution:")
print(df["score"].value_counts().sort_index())

Duplicate review IDs: 0

Platform distribution:
android_or_ios
android    26925
ios         1923
Name: count, dtype: int64

Rating distribution:
score
1    11087
2     1503
3     1252
4     1635
5    13371
Name: count, dtype: int64


In [8]:
df[["score", "content", "android_or_ios", "at"]].sample(
    min(10, len(df)),
    random_state=42
)

,score,content,android_or_ios,at
24847,5,works great,android,1/23/2026 17:38
9071,5,it's so fast and easy to send and receive money,android,7/8/2026 7:31
14881,1,"Long time user of PayPal for 5+ years, suspended my account when applying for the PayPal debit card with no explanation or appeal option. Contacte...",android,4/29/2026 23:43
6188,5,fast,android,8/11/2026 20:16
27052,2,I love paypal but why fix something that wasnât broke? I HATE the new dark look. I canât see it. Plus I canât find anything I need. Please m...,ios,10/22/2026 3:55
20653,5,updates are more fluid between venmo and paypal open up alot now,android,3/6/2026 2:07
14202,5,amazing,android,5/7/2026 13:22
11617,1,"I wouldn't even give a 1 star, They permanently disabled my account without a reason I couldn't even get a answer to why it was done specifically ...",android,6/6/2026 12:37
15817,2,have to pay fees when sending money,android,4/19/2026 21:31
5221,1,I have just updated it and now it is in a loop. I have to update the program and it takes me back to the update page. It keeps repeating this just...,android,8/22/2026 20:54


### Data Preprocessing

In [9]:
# Remove duplicate reviews
df = df.drop_duplicates(subset="reviewId").copy()

# Convert dates
df["at"] = pd.to_datetime(df["at"], errors="coerce", utc=True)

# Convert ratings to numeric
df["score"] = pd.to_numeric(df["score"], errors="coerce")

# Clean review text
df["content"] = df["content"].fillna("").astype(str).str.strip()

# Clean platform information
df["android_or_ios"] = (
    df["android_or_ios"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Remove records without actual review text
df = df[df["content"].str.len() > 0].copy()

# Extract date components
df["review_date"] = df["at"].dt.date
df["review_month"] = df["at"].dt.to_period("M").astype(str)
df["review_year"] = df["at"].dt.year

# Review length
df["review_length"] = df["content"].str.split().str.len()

df.reset_index(drop=True, inplace=True)

df.head()

,reviewId,score,content,at,android_or_ios,thumbsUpCount,reviewCreatedVersion,replyContent,review_date,review_month,review_year,review_length
0,8938a0da-4273-4019-865c-16ebf97e251a,5,brilliant,2026-10-26 15:18:00+00:00,android,0.0,10.14.0,NaN,2026-10-26,2026-10,2026,1
1,a5a628f9-122a-4819-823f-dedb4777f52d,1,Too much Charges ??,2026-10-26 14:55:00+00:00,android,0.0,10.18.0,NaN,2026-10-26,2026-10,2026,4
2,a75b079c-cafb-4ba2-a662-8992d66ec888,1,"Your agents are dumb, trash and such a big LIARS! Also the app and website and everything in this app is a trash. I can't link my bank card!!! It ...",2026-10-26 14:33:00+00:00,android,0.0,10.18.0,NaN,2026-10-26,2026-10,2026,84
3,719c5ac0-6fa8-4771-a46b-967874ef97eb,5,PayPal has gotten even better. Now I can choose how much I want to transfer. I don't have to send it all.,2026-10-26 13:30:00+00:00,android,0.0,10.18.0,NaN,2026-10-26,2026-10,2026,22
4,83993c40-362f-4f55-a0c0-6ce36e6f62ef,1,hacked,2026-10-26 13:28:00+00:00,android,0.0,10.18.0,NaN,2026-10-26,2026-10,2026,1


### EDA

In [10]:
fig = px.histogram(
    df,
    x="score",
    color="android_or_ios",
    barmode="group",
    title="PayPal Review Rating Distribution",
    labels={"score": "Star Rating", "count": "Number of Reviews"},
    category_orders={"score": [1, 2, 3, 4, 5]}
)

fig.show()

In [11]:
platform_rating = (
    df.groupby("android_or_ios")["score"]
    .agg(["count", "mean", "median"])
    .reset_index()
)

platform_rating

,android_or_ios,count,mean,median
0,android,26923,3.277755,4.0
1,ios,1923,1.553302,1.0


In [12]:
monthly_reviews = (
    df.dropna(subset=["at"])
    .groupby("review_month")
    .agg(
        total_reviews=("reviewId", "count"),
        avg_rating=("score", "mean")
    )
    .reset_index()
)

fig = px.line(
    monthly_reviews,
    x="review_month",
    y="total_reviews",
    markers=True,
    title="Monthly PayPal Review Volume"
)

fig.show()

In [13]:
fig = px.line(
    monthly_reviews,
    x="review_month",
    y="avg_rating",
    markers=True,
    title="Average Customer Rating Over Time"
)

fig.update_yaxes(range=[1, 5])
fig.show()

In [14]:
version_analysis = (
    df.groupby("reviewCreatedVersion")
    .agg(
        total_reviews=("reviewId", "count"),
        avg_rating=("score", "mean")
    )
    .query("total_reviews >= 10")
    .sort_values("total_reviews", ascending=False)
    .reset_index()
)

version_analysis.head(20)

,reviewCreatedVersion,total_reviews,avg_rating
0,8.107.0,4544,3.458407
1,8.98.1,3020,3.599669
2,8.105.1,2003,3.480280
3,8.103.0,1444,3.467452
4,8.100.1,1337,3.569933
5,8.102.0,1119,3.506702
6,8.104.1,814,3.417690
7,8.97.0,788,3.527919
8,10.14.0,721,2.285714
9,8.106.0,572,3.076923


### Sentiment Analysis

In [15]:
# !pip install vaderSentiment
# !pip install -U transformers torch pandas plotly

In [16]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    scores = analyzer.polarity_scores(text)
    compound = scores["compound"]

    if compound >= 0.05:
        return "Positive"
    elif compound <= -0.05:
        return "Negative"
    else:
        return "Neutral"

df["sentiment"] = df["content"].apply(get_sentiment)

In [17]:
sentiment_counts = (
    df["sentiment"]
    .value_counts()
    .reset_index()
)

fig = px.pie(
    sentiment_counts,
    names="sentiment",
    values="count",
    title="Customer Sentiment Distribution",
    hole=0.4
)

fig.show()

### Complaint categorisation

In [18]:
# Understand why customers are unhappy.

# Initially explore these categories:
# 1. Payment failures
# 2. Refund delays
# 3. Account restrictions
# 4. Login and authentication
# 5. Fraud and unauthorised transactions
# 6. Disputes and chargebacks
# 7. Customer support
# 8. Bank and card linking
# 9. International transfers
# 10. Other

categories = {
    "Payment Issues": [
        "payment failed", "transaction failed",
        "unable to pay", "payment pending"
    ],
    "Refund Issues": [
        "refund", "money back", "refund pending"
    ],
    "Account Restrictions": [
        "account limited", "account restricted",
        "account blocked", "frozen account"
    ],
    "Login Issues": [
        "login", "sign in", "password", "verification"
    ],
    "Fraud & Security": [
        "unauthorized", "unauthorised",
        "hacked", "stolen", "fraud", "scam"
    ],
    "Disputes": [
        "dispute", "chargeback", "claim"
    ],
    "Customer Support": [
        "customer service", "support", "no response",
        "representative"
    ],
    "Bank & Card Issues": [
        "bank account", "debit card", "credit card",
        "link card"
    ]
}

In [19]:
def classify_review(text):
    text = text.lower()

    matched_categories = []

    for category, keywords in categories.items():
        if any(keyword in text for keyword in keywords):
            matched_categories.append(category)

    return matched_categories if matched_categories else ["Other"]


df["complaint_category"] = df["content"].apply(classify_review)

# Convert multi-label categories into separate rows
category_df = df.explode("complaint_category")

category_counts = (
    category_df["complaint_category"]
    .value_counts()
    .reset_index()
)

category_counts.columns = ["Category", "Review Count"]

px.bar(
    category_counts,
    x="Review Count",
    y="Category",
    orientation="h",
    title="Most Frequently Reported Customer Issues"
).show()

### New Technique

In [20]:
import torch
from transformers import pipeline

# 1. Define complaint categories and descriptions
categories = {
    "Payment Issues": "Problems with sending, receiving, processing or completing payments and transactions",
    "Account Restrictions": "Account limitations, account suspension, frozen accounts or blocked accounts",
    "Disputes": "Transaction disputes, chargebacks, claims and disagreements over payments",
    "Refund Issues": "Delayed refunds, missing refunds, refund processing and money not returned",
    "Bank & Card Issues": "Problems linking bank accounts, debit cards, credit cards or withdrawing money",
    "Fraud & Security": "Unauthorised transactions, scams, hacking, stolen money and security concerns",
    "Login & Verification": "Login failures, password problems, OTP issues, identity verification and account access",
    "Customer Support": "Poor customer service, unresolved complaints, unresponsive support and difficulty contacting agents",
    "App Performance": "Application crashes, bugs, loading problems, technical errors and slow performance",
    "Fees & Charges": "Unexpected fees, transaction charges, currency conversion fees and hidden costs"
}

# 2. Load model on GPU
classifier = pipeline(
    "zero-shot-classification",
    model="valhalla/distilbart-mnli-12-1",
    device=0
)

# 3. Separate category names and descriptions
candidate_labels = list(categories.values())

description_to_category = {
    description: category
    for category, description in categories.items()
}

# 4. Verify configuration
print("Model loaded on:", torch.cuda.get_device_name(0))
print("CUDA available:", torch.cuda.is_available())
print("Number of complaint categories:", len(candidate_labels))

W1104 19:45:30.068000 17808 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 231/231 [00:00<00:00, 17141.11it/s]


Model loaded on: NVIDIA GeForce RTX 4050 Laptop GPU
CUDA available: True
Number of complaint categories: 10


In [22]:
import pandas as pd
import torch
from tqdm.auto import tqdm

# Select low-rated reviews
issue_df = df[df["score"] <= 3].copy()

# Remove empty reviews
issue_df = issue_df[
    issue_df["content"].notna() &
    (issue_df["content"].str.strip() != "")
].copy()

# Reset index
issue_df.reset_index(drop=True, inplace=True)

# Prepare review texts
texts = issue_df["content"].astype(str).tolist()

# GPU batch configuration
BATCH_SIZE = 8
THRESHOLD = 0.35

all_categories = []

# Process reviews in batches
for start in tqdm(range(0, len(texts), BATCH_SIZE)):

    batch_texts = texts[start:start + BATCH_SIZE]

    results = classifier(
        batch_texts,
        candidate_labels=candidate_labels,
        hypothesis_template="This customer review describes {}.",
        multi_label=True,
        batch_size=BATCH_SIZE,
        truncation=True,
        max_length=256
    )

    for result in results:

        matched = [
            description_to_category[label]
            for label, score in zip(
                result["labels"],
                result["scores"]
            )
            if score >= THRESHOLD
        ]

        if not matched:
            matched = ["Other"]

        all_categories.append(matched)

# Store predictions
issue_df["complaint_category"] = all_categories

print("Classification completed!")

100%|██████████| 1731/1731 [11:44<00:00,  2.46it/s]

Classification completed!


In [24]:
category_df = issue_df.explode("complaint_category")

category_counts = (
    category_df["complaint_category"]
    .value_counts()
    .reset_index()
)

category_counts.columns = ["Category", "Review Count"]

category_counts

,Category,Review Count
0,Customer Support,6460
1,Other,5379
2,Payment Issues,3836
3,Fees & Charges,3492
4,App Performance,3247
5,Refund Issues,3154
6,Bank & Card Issues,3050
7,Login & Verification,2826
8,Fraud & Security,2443
9,Account Restrictions,2099


In [26]:
import plotly.express as px

fig = px.bar(
    category_counts.sort_values("Review Count"),
    x="Review Count",
    y="Category",
    orientation="h",
    title="Most Frequently Reported Customer Issues",
    text="Review Count",
    color="Review Count",
    color_continuous_scale="Blues"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Number of Reviews",
    yaxis_title="Category",
    showlegend=False
)

fig.show()

issue_df[
    ["score", "content", "complaint_category"]
].sample(30, random_state=42)

,score,content,complaint_category
12960,1,There is no where to log out,[Other]
11585,3,I can't change my password . Why don't you put an option to change password like you have the option to change phone number and email ?,[Other]
4128,1,will not accept my birthday onfo,[Other]
4786,1,"terrible app. terrible customer service and they can't fix and issues you have. don't bother, just do everything through the bank.",[Customer Support]
1599,1,my account have a limitation? and you resolved in 180 days? the heck is that? you limiting may account because theres have a money send on the acc...,[Other]
4004,1,"awful app. account recovery is the worst thing i've ever used period. to contact customer support, you have to be logged in, which is the reason w...","[Login & Verification, Fraud & Security, Customer Support, Payment Issues, App Performance, Bank & Card Issues, Account Restrictions]"
1684,1,l have never claimed in this app please guide me,[Other]
4974,1,"I just linked my account and every time I try to change my phone number it sends a code, asks for the code, and then gives me another code. its a ...","[Customer Support, Payment Issues]"
4139,1,"This app have been little frustrating, i have been trying to open it but is not working .... Thanks","[App Performance, Login & Verification, Bank & Card Issues, Payment Issues]"
8009,2,"Genuinely obnoxious to use, but one of the few P2P payment apps that don't take insane cuts. They completely neglect your concerns, yet go wild on...",[Customer Support]


In [25]:
print(
    "GPU Memory Used:",
    round(torch.cuda.memory_allocated(0) / 1024**3, 2),
    "GB"
)

GPU Memory Used: 0.84 GB


In [34]:
# Work on a copy. Preserve your original issue_df.
sentinel_df = issue_df.copy()

print("Dataset shape:", sentinel_df.shape)

print("\nColumn names:")
print(sentinel_df.columns.tolist())

print("\nData types:")
display(sentinel_df.dtypes.to_frame("Data Type"))

print("\nMissing values:")
display(
    sentinel_df.isnull().sum()
    .to_frame("Missing Values")
    .sort_values("Missing Values", ascending=False)
)

print("\nDuplicate review IDs:", sentinel_df["reviewId"].duplicated().sum())

print("\nDuplicate review text:", sentinel_df["content"].duplicated().sum())

Dataset shape: (13842, 14)

Column names:
['reviewId', 'score', 'content', 'at', 'android_or_ios', 'thumbsUpCount', 'reviewCreatedVersion', 'replyContent', 'review_date', 'review_month', 'review_year', 'review_length', 'sentiment', 'complaint_category']

Data types:


,Data Type
reviewId,str
score,int64
content,str
at,"datetime64[us, UTC]"
android_or_ios,string
thumbsUpCount,float64
reviewCreatedVersion,str
replyContent,str
review_date,object
review_month,str



Missing values:


,Missing Values
replyContent,13765
reviewCreatedVersion,2921
thumbsUpCount,1732
content,0
score,0
reviewId,0
android_or_ios,0
at,0
review_date,0
review_month,0



Duplicate review IDs: 0

Duplicate review text: 285


In [ ]:
# # Display 15 representative reviews
# sample_reviews = sentinel_df[
#     ["score", "content", "android_or_ios",
#      "reviewCreatedVersion", "complaint_category"]
# ].sample(
#     min(15, len(sentinel_df)),
#     random_state=42
# )

# pd.set_option("display.max_colwidth", 250)

# display(sample_reviews)

,score,content,android_or_ios,reviewCreatedVersion,complaint_category
12960,1,There is no where to log out,ios,10.0.1,[Other]
11585,3,I can't change my password . Why don't you put an option to change password like you have the option to change phone number and email ?,android,NaN,[Other]
4128,1,will not accept my birthday onfo,android,NaN,[Other]
4786,1,"terrible app. terrible customer service and they can't fix and issues you have. don't bother, just do everything through the bank.",android,8.107.0,[Customer Support]
1599,1,my account have a limitation? and you resolved in 180 days? the heck is that? you limiting may account because theres have a money send on the account? why there's no warnings?,android,10.13.0,[Other]
4004,1,"awful app. account recovery is the worst thing i've ever used period. to contact customer support, you have to be logged in, which is the reason why i'm trying to contact customer support in the first place. phone number verification doesn't work...",android,8.107.0,"[Login & Verification, Fraud & Security, Customer Support, Payment Issues, App Performance, Bank & Card Issues, Account Restrictions]"
1684,1,l have never claimed in this app please guide me,android,NaN,[Other]
4974,1,"I just linked my account and every time I try to change my phone number it sends a code, asks for the code, and then gives me another code. its a full loop and its so annoying. whenever I actually get somewhere it says theres an error on there en...",android,NaN,"[Customer Support, Payment Issues]"
4139,1,"This app have been little frustrating, i have been trying to open it but is not working .... Thanks",android,NaN,"[App Performance, Login & Verification, Bank & Card Issues, Payment Issues]"
8009,2,"Genuinely obnoxious to use, but one of the few P2P payment apps that don't take insane cuts. They completely neglect your concerns, yet go wild on the other end, fully locking your account just for filing a dispute.",android,8.62.1,[Customer Support]


In [36]:
print("Platform distribution:")
display(
    sentinel_df["android_or_ios"]
    .value_counts(dropna=False)
    .to_frame("Review Count")
)

print("\nTop 20 app versions:")
display(
    sentinel_df["reviewCreatedVersion"]
    .value_counts(dropna=False)
    .head(20)
    .to_frame("Review Count")
)

print("\nReview timeline:")
display(
    sentinel_df.groupby(
        ["review_year", "review_month"]
    ).size().reset_index(name="Review Count")
    .sort_values(["review_year", "review_month"])
)

Platform distribution:


,Review Count
android_or_ios,
android,12110
ios,1732



Top 20 app versions:


,Review Count
reviewCreatedVersion,
NaN,2921
8.107.0,1829
8.98.1,1090
8.105.1,798
8.103.0,580
10.14.0,520
8.100.1,491
8.102.0,431
10.12.0,379



Review timeline:


,review_year,review_month,Review Count
0,2026,2026-01,1191
1,2026,2026-02,1180
2,2026,2026-03,1293
3,2026,2026-04,1326
4,2026,2026-05,1232
5,2026,2026-06,1207
6,2026,2026-07,1270
7,2026,2026-08,1406
8,2026,2026-09,2044
9,2026,2026-10,1693


In [37]:
def clean_review_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Remove excessive whitespace and line breaks
    text = re.sub(r"\s+", " ", text)

    return text.strip()


sentinel_df["review_text"] = sentinel_df["content"].apply(
    clean_review_text
)

# Remove empty reviews
sentinel_df = sentinel_df[
    sentinel_df["review_text"].str.len() > 0
].copy()

sentinel_df.reset_index(drop=True, inplace=True)

print("Prepared dataset shape:", sentinel_df.shape)

display(
    sentinel_df[
        ["content", "review_text"]
    ].head(10)
)

Prepared dataset shape: (13842, 15)


,content,review_text
0,Too much Charges ??,Too much Charges ??
1,"Your agents are dumb, trash and such a big LIARS! Also the app and website and everything in this app is a trash. I can't link my bank card!!! It always shows an error message whenever I try! Now, I contacted an AGENT only to be told by pure lies...","Your agents are dumb, trash and such a big LIARS! Also the app and website and everything in this app is a trash. I can't link my bank card!!! It always shows an error message whenever I try! Now, I contacted an AGENT only to be told by pure lies..."
2,hacked,hacked
3,app not compatible with tablet devices,app not compatible with tablet devices
4,I will give 1 star only why? because when I reset my password cus I forgot it they only give 5seconds to write the password how can u write it so quick when it takes more than 5 secs to wait for the code..,I will give 1 star only why? because when I reset my password cus I forgot it they only give 5seconds to write the password how can u write it so quick when it takes more than 5 secs to wait for the code..
5,useless app,useless app
6,worst banking app I've ever had the misfortune of using,worst banking app I've ever had the misfortune of using
7,constantly fails login by failing authenticator code.,constantly fails login by failing authenticator code.
8,scam,scam
9,????? ????????? ??????????????,????? ????????? ??????????????


In [38]:
def normalise_categories(value):

    if isinstance(value, list):
        return value

    if isinstance(value, (tuple, set)):
        return list(value)

    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)

            if isinstance(parsed, (list, tuple, set)):
                return list(parsed)

        except (ValueError, SyntaxError):
            pass

        return [value] if value.strip() else []

    return []


sentinel_df["complaint_category"] = (
    sentinel_df["complaint_category"]
    .apply(normalise_categories)
)

print("Category parsing completed.")

display(
    sentinel_df[
        ["review_text", "complaint_category"]
    ].head(10)
)

Category parsing completed.


,review_text,complaint_category
0,Too much Charges ??,[Fees & Charges]
1,"Your agents are dumb, trash and such a big LIARS! Also the app and website and everything in this app is a trash. I can't link my bank card!!! It always shows an error message whenever I try! Now, I contacted an AGENT only to be told by pure lies...","[Bank & Card Issues, Payment Issues, Customer Support, Login & Verification, App Performance, Disputes, Fees & Charges, Fraud & Security, Refund Issues, Account Restrictions]"
2,hacked,[Fraud & Security]
3,app not compatible with tablet devices,[Other]
4,I will give 1 star only why? because when I reset my password cus I forgot it they only give 5seconds to write the password how can u write it so quick when it takes more than 5 secs to wait for the code..,[Other]
5,useless app,"[Customer Support, App Performance]"
6,worst banking app I've ever had the misfortune of using,"[App Performance, Fees & Charges, Fraud & Security, Customer Support, Login & Verification, Refund Issues]"
7,constantly fails login by failing authenticator code.,[Other]
8,scam,"[Fraud & Security, Fees & Charges, Customer Support, Refund Issues]"
9,????? ????????? ??????????????,[Customer Support]


In [39]:
PRODUCT_TAXONOMY = {
    "Wallet": {
        "Account & Balance": [
            "Balance Visibility",
            "Available vs Pending Balance",
            "Account Limits"
        ],
        "Funding Sources": [
            "Add Bank Account",
            "Bank Account Verification",
            "Add Debit Card",
            "Add Credit Card",
            "Remove Funding Source"
        ]
    },

    "Payments": {
        "Send Money": [
            "Recipient Selection",
            "Payment Submission",
            "Payment Pending",
            "Payment Failure",
            "Duplicate Payment"
        ],
        "Receive Money": [
            "Incoming Payment",
            "Payment Confirmation",
            "Received Money Not Reflected"
        ],
        "Checkout": [
            "PayPal Checkout Button",
            "Merchant Checkout Redirection",
            "Checkout Authentication"
        ]
    },

    "Money Movement": {
        "Withdrawals": [
            "Bank Withdrawal",
            "Debit Card Withdrawal",
            "Withdrawal Pending",
            "Withdrawal Failure"
        ],
        "Refunds": [
            "Initiate Refund",
            "Refund Status Tracking",
            "Refund Not Received"
        ]
    },

    "Dispute Management": {
        "Resolution Centre": [
            "Create Dispute",
            "Upload Evidence",
            "Track Dispute Status",
            "Communicate on Dispute"
        ],
        "Chargebacks": [
            "Submit Chargeback Response",
            "Chargeback Evidence",
            "Appeal Dispute Outcome"
        ]
    },

    "Identity & Security": {
        "Authentication": [
            "Login",
            "Password Reset",
            "OTP / MFA Verification",
            "Session Management"
        ],
        "Identity Verification": [
            "Identity Document Submission",
            "Identity Verification Status",
            "Account Verification",
            "Account Restriction"
        ]
    },

    "Pricing & Currency": {
        "Fees & Currency": [
            "Transaction Fees",
            "Currency Conversion",
            "Unexpected Charges"
        ]
    },

    "Mobile Application": {
        "App Experience": [
            "Application Crash",
            "Application Loading",
            "Navigation",
            "Notifications",
            "App Compatibility",
            "Application Update"
        ]
    }
}

In [40]:
taxonomy_rows = []

for domain, modules in PRODUCT_TAXONOMY.items():

    for module, features in modules.items():

        for feature in features:

            taxonomy_rows.append({
                "product_domain": domain,
                "product_module": module,
                "product_feature": feature
            })

taxonomy_df = pd.DataFrame(taxonomy_rows)

print("Total defined features:", len(taxonomy_df))

display(taxonomy_df)

Total defined features: 50


,product_domain,product_module,product_feature
0,Wallet,Account & Balance,Balance Visibility
1,Wallet,Account & Balance,Available vs Pending Balance
2,Wallet,Account & Balance,Account Limits
3,Wallet,Funding Sources,Add Bank Account
4,Wallet,Funding Sources,Bank Account Verification
5,Wallet,Funding Sources,Add Debit Card
6,Wallet,Funding Sources,Add Credit Card
7,Wallet,Funding Sources,Remove Funding Source
8,Payments,Send Money,Recipient Selection
9,Payments,Send Money,Payment Submission


In [41]:
UNKNOWN = "Unknown / Insufficient information"

ALLOWED_FAILURE_MODES = [
    "Action unavailable",
    "Action rejected",
    "Action fails during submission",
    "Action remains pending",
    "Action completed but not reflected",
    "Unexpected status or outcome",
    "Incorrect information displayed",
    "Unexpected charge or deduction",
    "Application crashes",
    "Application does not load",
    "Performance or latency issue",
    "Authentication failure",
    "Verification failure",
    "Customer unable to locate feature",
    "Other",
    UNKNOWN
]

category_samples = (
    sentinel_df
    .explode("complaint_category")
    .groupby("complaint_category", dropna=False)
    .agg(
        review_count=("review_text", "count"),
        examples=("review_text", lambda x: list(x.head(3)))
    )
    .reset_index()
    .sort_values("review_count", ascending=False)
)

display(category_samples)

,complaint_category,review_count,examples
3,Customer Support,6460,"[Your agents are dumb, trash and such a big LIARS! Also the app and website and everything in this app is a trash. I can't link my bank card!!! It always shows an error message whenever I try! Now, I contacted an AGENT only to be told by pure lie..."
8,Other,5379,"[app not compatible with tablet devices, I will give 1 star only why? because when I reset my password cus I forgot it they only give 5seconds to write the password how can u write it so quick when it takes more than 5 secs to wait for the code....."
9,Payment Issues,3836,"[Your agents are dumb, trash and such a big LIARS! Also the app and website and everything in this app is a trash. I can't link my bank card!!! It always shows an error message whenever I try! Now, I contacted an AGENT only to be told by pure lie..."
5,Fees & Charges,3492,"[Too much Charges ??, Your agents are dumb, trash and such a big LIARS! Also the app and website and everything in this app is a trash. I can't link my bank card!!! It always shows an error message whenever I try! Now, I contacted an AGENT only t..."
1,App Performance,3247,"[Your agents are dumb, trash and such a big LIARS! Also the app and website and everything in this app is a trash. I can't link my bank card!!! It always shows an error message whenever I try! Now, I contacted an AGENT only to be told by pure lie..."
10,Refund Issues,3154,"[Your agents are dumb, trash and such a big LIARS! Also the app and website and everything in this app is a trash. I can't link my bank card!!! It always shows an error message whenever I try! Now, I contacted an AGENT only to be told by pure lie..."
2,Bank & Card Issues,3050,"[Your agents are dumb, trash and such a big LIARS! Also the app and website and everything in this app is a trash. I can't link my bank card!!! It always shows an error message whenever I try! Now, I contacted an AGENT only to be told by pure lie..."
7,Login & Verification,2826,"[Your agents are dumb, trash and such a big LIARS! Also the app and website and everything in this app is a trash. I can't link my bank card!!! It always shows an error message whenever I try! Now, I contacted an AGENT only to be told by pure lie..."
6,Fraud & Security,2443,"[Your agents are dumb, trash and such a big LIARS! Also the app and website and everything in this app is a trash. I can't link my bank card!!! It always shows an error message whenever I try! Now, I contacted an AGENT only to be told by pure lie..."
0,Account Restrictions,2099,"[Your agents are dumb, trash and such a big LIARS! Also the app and website and everything in this app is a trash. I can't link my bank card!!! It always shows an error message whenever I try! Now, I contacted an AGENT only to be told by pure lie..."


In [45]:
import hashlib

# Version our data contract so future changes are traceable.
CDM_VERSION = "1.0"

# Standard identifiers and metadata shared across entities.
COMMON_FIELDS = [
    "record_id",
    "source_system",
    "source_record_id",
    "schema_version",
    "ingested_at",
    "event_timestamp"
]

# Entity schemas: fields are standardised across source systems.
CDM_SCHEMAS = {

    "complaints": COMMON_FIELDS + [
        "customer_ref",
        "review_rating",
        "review_text",
        "platform",
        "app_version",
        "complaint_category",
        "product_domain",
        "product_module",
        "product_feature",
        "failure_mode",
        "correlation_id",
        "transaction_id",
        "dispute_id"
    ],

    "transactions": COMMON_FIELDS + [
        "transaction_id",
        "customer_ref",
        "amount",
        "currency",
        "transaction_type",
        "transaction_status",
        "payment_method",
        "correlation_id"
    ],

    "transaction_events": COMMON_FIELDS + [
        "transaction_id",
        "event_type",
        "event_status",
        "error_code",
        "correlation_id"
    ],

    "disputes": COMMON_FIELDS + [
        "dispute_id",
        "transaction_id",
        "customer_ref",
        "dispute_reason",
        "dispute_status",
        "lifecycle_stage",
        "evidence_status",
        "response_due_date"
    ],

    "telemetry_events": COMMON_FIELDS + [
        "correlation_id",
        "transaction_id",
        "api_endpoint",
        "http_status",
        "error_code",
        "error_message",
        "latency_ms",
        "app_version",
        "platform"
    ],

    "engineering_issues": COMMON_FIELDS + [
        "engineering_issue_id",
        "product_domain",
        "product_module",
        "product_feature",
        "failure_mode",
        "severity",
        "priority",
        "issue_status",
        "root_cause",
        "resolution_summary"
    ],

    "releases": COMMON_FIELDS + [
        "release_id",
        "app_version",
        "deployment_timestamp",
        "release_description",
        "engineering_issue_id"
    ]
}

# Create empty canonical tables.
sentinel_data = {
    entity: pd.DataFrame(columns=columns)
    for entity, columns in CDM_SCHEMAS.items()
}

for entity, table in sentinel_data.items():
    print(f"{entity}: {table.shape}")

complaints: (0, 19)
transactions: (0, 14)
transaction_events: (0, 11)
disputes: (0, 14)
telemetry_events: (0, 15)
engineering_issues: (0, 16)
releases: (0, 11)


In [46]:
CDM_SCHEMAS["complaints"].extend([
    "sentiment",
    "thumbs_up_count"
])

In [47]:
def generate_record_id(source_system, source_record_id, fallback_text=""):

    identity = (
        f"{source_system}|"
        f"{source_record_id}|"
        f"{fallback_text}"
    )

    return hashlib.sha256(
        identity.encode("utf-8")
    ).hexdigest()[:20]


def identify_source(platform):

    value = str(platform).lower()

    if "android" in value:
        return "google_play"

    elif "ios" in value:
        return "apple_app_store"

    return "unknown_app_store"


def map_public_reviews_to_cdm(df):

    records = []

    for _, row in df.iterrows():

        source_system = identify_source(
            row.get("android_or_ios")
        )

        source_record_id = row.get("reviewId")

        if pd.isna(source_record_id):
            source_record_id = None
        else:
            source_record_id = str(source_record_id)

        review_text = str(row.get("review_text", ""))

        # Generate an internal ID without fabricating a source ID.
        record_id = generate_record_id(
            source_system,
            source_record_id,
            fallback_text=(
                review_text if source_record_id is None else ""
            )
        )

        # Prefer review_date, then fall back to the original date.
        timestamp_value = row.get("review_date")

        if pd.isna(timestamp_value):
            timestamp_value = row.get("at")

        event_timestamp = pd.to_datetime(
            timestamp_value,
            errors="coerce",
            utc=True
        )

        categories = row.get("complaint_category", [])

        if not isinstance(categories, list):
            categories = normalise_categories(categories)

        record = {
            "record_id": record_id,
            "source_system": source_system,
            "source_record_id": source_record_id,
            "schema_version": CDM_VERSION,
            "ingested_at": pd.Timestamp.now(tz="UTC"),
            "event_timestamp": event_timestamp,

            # No customer or payment identifiers are available
            # from the public review dataset.
            "customer_ref": None,
            "transaction_id": None,
            "dispute_id": None,
            "correlation_id": None,

            "review_rating": row.get("score"),
            "review_text": review_text,
            "platform": row.get("android_or_ios"),
            "app_version": row.get("reviewCreatedVersion"),
            "complaint_category": json.dumps(categories),

            # These will be populated in the AI extraction stage.
            "product_domain": None,
            "product_module": None,
            "product_feature": None,
            "failure_mode": None,

            "sentiment": row.get("sentiment"),
            "thumbs_up_count": row.get("thumbsUpCount")
        }

        records.append(record)

    return pd.DataFrame(
        records,
        columns=CDM_SCHEMAS["complaints"]
    )


# Create the canonical complaints table.
sentinel_data["complaints"] = map_public_reviews_to_cdm(
    sentinel_df
)

complaints_cdm = sentinel_data["complaints"]

print("Canonical complaints shape:", complaints_cdm.shape)

display(complaints_cdm.head())

Canonical complaints shape: (13842, 21)


,record_id,source_system,source_record_id,schema_version,ingested_at,event_timestamp,customer_ref,review_rating,review_text,platform,app_version,complaint_category,product_domain,product_module,product_feature,failure_mode,correlation_id,transaction_id,dispute_id,sentiment,thumbs_up_count
0,3c4d15d6e97228d70249,google_play,a5a628f9-122a-4819-823f-dedb4777f52d,1.0,2026-11-05 10:06:06.265790+00:00,2026-10-26 00:00:00+00:00,None,1,Too much Charges ??,android,10.18.0,"[""Fees & Charges""]",None,None,None,None,None,None,None,Negative,0.0
1,b86ba5f1b75ac3d082b6,google_play,a75b079c-cafb-4ba2-a662-8992d66ec888,1.0,2026-11-05 10:06:06.266564+00:00,2026-10-26 00:00:00+00:00,None,1,"Your agents are dumb, trash and such a big LIARS! Also the app and website and everything in this app is a trash. I can't link my bank card!!! It always shows an error message whenever I try! Now, I contacted an AGENT only to be told by pure lies...",android,10.18.0,"[""Bank & Card Issues"", ""Payment Issues"", ""Customer Support"", ""Login & Verification"", ""App Performance"", ""Disputes"", ""Fees & Charges"", ""Fraud & Security"", ""Refund Issues"", ""Account Restrictions""]",None,None,None,None,None,None,None,Negative,0.0
2,885b45d14b25869bfb65,google_play,83993c40-362f-4f55-a0c0-6ce36e6f62ef,1.0,2026-11-05 10:06:06.266708+00:00,2026-10-26 00:00:00+00:00,None,1,hacked,android,10.18.0,"[""Fraud & Security""]",None,None,None,None,None,None,None,Negative,0.0
3,75888340076f8b988c76,google_play,faeb8166-6c7c-41b2-92b8-abcc99232b05,1.0,2026-11-05 10:06:06.266817+00:00,2026-10-26 00:00:00+00:00,None,1,app not compatible with tablet devices,android,8.107.0,"[""Other""]",None,None,None,None,None,None,None,Neutral,0.0
4,6585f2080dd1b27f1f7b,google_play,c07892f6-b7a5-495d-83bd-6c2094d31244,1.0,2026-11-05 10:06:06.266910+00:00,2026-10-26 00:00:00+00:00,None,1,I will give 1 star only why? because when I reset my password cus I forgot it they only give 5seconds to write the password how can u write it so quick when it takes more than 5 secs to wait for the code..,android,10.18.0,"[""Other""]",None,None,None,None,None,None,None,Neutral,0.0


In [48]:
# 1. Check schema consistency
expected_columns = set(CDM_SCHEMAS["complaints"])
actual_columns = set(complaints_cdm.columns)

print("Missing columns:", expected_columns - actual_columns)
print("Unexpected columns:", actual_columns - expected_columns)

# 2. Check record ID uniqueness
print(
    "\nDuplicate internal record IDs:",
    complaints_cdm["record_id"].duplicated().sum()
)

# 3. Check source distribution
display(
    complaints_cdm["source_system"]
    .value_counts(dropna=False)
    .to_frame("Record Count")
)

# 4. Check available linking identifiers
linkage_fields = [
    "customer_ref",
    "transaction_id",
    "dispute_id",
    "correlation_id"
]

linkage_summary = pd.DataFrame({
    "Field": linkage_fields,
    "Available Records": [
        complaints_cdm[field].notna().sum()
        for field in linkage_fields
    ]
})

display(linkage_summary)

# 5. Check timestamp coverage
print(
    "\nRecords with valid timestamps:",
    complaints_cdm["event_timestamp"].notna().sum()
)

print(
    "Total records:",
    len(complaints_cdm)
)

Missing columns: set()
Unexpected columns: set()

Duplicate internal record IDs: 0


,Record Count
source_system,
google_play,12110
apple_app_store,1732


,Field,Available Records
0,customer_ref,0
1,transaction_id,0
2,dispute_id,0
3,correlation_id,0



Records with valid timestamps: 13842
Total records: 13842


In [49]:
paypal_transaction_example = pd.DataFrame({
    "txn_id": ["TXN001", "TXN002"],
    "payment_amount": [2500, 1200],
    "currency_code": ["INR", "INR"],
    "payment_state": ["COMPLETED", "PENDING"],
    "api_correlation": ["CORR001", "CORR002"]
})

display(paypal_transaction_example)

,txn_id,payment_amount,currency_code,payment_state,api_correlation
0,TXN001,2500,INR,COMPLETED,CORR001
1,TXN002,1200,INR,PENDING,CORR002


In [50]:
def map_paypal_transactions(df):

    mapped = pd.DataFrame({
        "transaction_id": df["txn_id"],
        "amount": pd.to_numeric(
            df["payment_amount"],
            errors="coerce"
        ),
        "currency": df["currency_code"],
        "transaction_status": df["payment_state"],
        "correlation_id": df["api_correlation"]
    })

    # Populate common metadata.
    mapped["record_id"] = mapped["transaction_id"].apply(
        lambda x: generate_record_id(
            "paypal_transaction",
            str(x)
        )
    )

    mapped["source_system"] = "paypal_transaction"
    mapped["source_record_id"] = df["txn_id"].astype(str)
    mapped["schema_version"] = CDM_VERSION
    mapped["ingested_at"] = pd.Timestamp.now(tz="UTC")
    mapped["event_timestamp"] = pd.NaT

    # These fields are not provided by this example.
    mapped["customer_ref"] = None
    mapped["transaction_type"] = None
    mapped["payment_method"] = None

    return mapped[CDM_SCHEMAS["transactions"]]


# Test the adapter.
test_transactions = map_paypal_transactions(
    paypal_transaction_example
)

display(test_transactions)

,record_id,source_system,source_record_id,schema_version,ingested_at,event_timestamp,transaction_id,customer_ref,amount,currency,transaction_type,transaction_status,payment_method,correlation_id
0,efb0db2e47f567bdf7ad,paypal_transaction,TXN001,1.0,2026-11-05 10:06:42.952411+00:00,NaT,TXN001,None,2500,INR,None,COMPLETED,None,CORR001
1,f4d77a43b2c016452888,paypal_transaction,TXN002,1.0,2026-11-05 10:06:42.952411+00:00,NaT,TXN002,None,1200,INR,None,PENDING,None,CORR002


### Create a Database with existing data

In [51]:
import sqlite3
import pandas as pd
import json
from pathlib import Path

# Create a project data directory
DATA_DIR = Path("sentinel_data")
DATA_DIR.mkdir(exist_ok=True)

DB_PATH = DATA_DIR / "sentinel.db"

# Connect to SQLite
conn = sqlite3.connect(DB_PATH)

# Prepare canonical complaint data
db_complaints = complaints_cdm.copy()

# Convert timestamps into ISO-format strings
for col in ["event_timestamp", "ingested_at"]:
    if col in db_complaints.columns:
        db_complaints[col] = pd.to_datetime(
            db_complaints[col],
            errors="coerce",
            utc=True
        ).astype("string")

# Store the complaint data
db_complaints.to_sql(
    "complaints",
    conn,
    if_exists="replace",
    index=False
)

# Store the product taxonomy
taxonomy_df.to_sql(
    "product_taxonomy",
    conn,
    if_exists="replace",
    index=False
)

print("Canonical data imported successfully.")
print("Database:", DB_PATH.resolve())

Canonical data imported successfully.
Database: C:\Users\shyam\python_test\PayPal_Hackathon\sentinel_data\sentinel.db


### Create the investigation workflow tables

In [52]:
with conn:

    # Engineering tickets
    conn.execute("""
        CREATE TABLE IF NOT EXISTS engineering_tickets (
            ticket_id TEXT PRIMARY KEY,
            ticket_title TEXT,
            product_domain TEXT,
            product_module TEXT,
            product_feature TEXT,
            failure_mode TEXT,
            severity TEXT,
            priority TEXT,
            technical_hypothesis TEXT,
            supporting_evidence TEXT,
            root_cause_status TEXT,
            ticket_status TEXT DEFAULT 'Open',
            resolution_summary TEXT,
            created_at TEXT,
            updated_at TEXT
        )
    """)

    # Investigation history
    conn.execute("""
        CREATE TABLE IF NOT EXISTS investigation_events (
            event_id INTEGER PRIMARY KEY AUTOINCREMENT,
            complaint_record_id TEXT,
            ticket_id TEXT,
            event_type TEXT,
            event_description TEXT,
            actor TEXT,
            event_timestamp TEXT
        )
    """)

    # Relationships across different data sources
    conn.execute("""
        CREATE TABLE IF NOT EXISTS issue_links (
            link_id INTEGER PRIMARY KEY AUTOINCREMENT,
            source_entity TEXT,
            source_record_id TEXT,
            target_entity TEXT,
            target_record_id TEXT,
            relationship_type TEXT,
            match_method TEXT,
            confidence REAL,
            created_at TEXT
        )
    """)

print("Workflow tables created.")

Workflow tables created.


In [53]:
# Cell 3 — Add database indexes

with conn:

    conn.execute("""
        CREATE INDEX IF NOT EXISTS idx_complaint_source
        ON complaints(source_system)
    """)

    conn.execute("""
        CREATE INDEX IF NOT EXISTS idx_complaint_version
        ON complaints(app_version)
    """)

    conn.execute("""
        CREATE INDEX IF NOT EXISTS idx_complaint_record
        ON complaints(record_id)
    """)

    conn.execute("""
        CREATE INDEX IF NOT EXISTS idx_ticket_status
        ON engineering_tickets(ticket_status)
    """)

    conn.execute("""
        CREATE INDEX IF NOT EXISTS idx_issue_links_source
        ON issue_links(source_entity, source_record_id)
    """)

print("Indexes created.")


# Cell 4 — Verify the database
tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
""", conn)

print("Database tables:")
display(tables)

print("\nComplaint records:")
print(
    pd.read_sql_query(
        "SELECT COUNT(*) AS total FROM complaints",
        conn
    )
)

print("\nSample complaints:")
display(
    pd.read_sql_query("""
        SELECT
            record_id,
            source_system,
            review_rating,
            review_text,
            app_version,
            product_domain
        FROM complaints
        LIMIT 5
    """, conn)
)

conn.close()

Indexes created.
Database tables:


,name
0,complaints
1,engineering_tickets
2,investigation_events
3,issue_links
4,product_taxonomy
5,sqlite_sequence



Complaint records:
   total
0  13842

Sample complaints:


,record_id,source_system,review_rating,review_text,app_version,product_domain
0,3c4d15d6e97228d70249,google_play,1,Too much Charges ??,10.18.0,None
1,b86ba5f1b75ac3d082b6,google_play,1,"Your agents are dumb, trash and such a big LIARS! Also the app and website and everything in this app is a trash. I can't link my bank card!!! It always shows an error message whenever I try! Now, I contacted an AGENT only to be told by pure lies...",10.18.0,None
2,885b45d14b25869bfb65,google_play,1,hacked,10.18.0,None
3,75888340076f8b988c76,google_play,1,app not compatible with tablet devices,8.107.0,None
4,6585f2080dd1b27f1f7b,google_play,1,I will give 1 star only why? because when I reset my password cus I forgot it they only give 5seconds to write the password how can u write it so quick when it takes more than 5 secs to wait for the code..,10.18.0,None


## PRODUCTION READY

In [9]:
# ============================================================
# CELL 1: ENVIRONMENT AND CONFIGURATION
# ============================================================

import os
import sys
import json
import hashlib
import platform
import warnings

from pathlib import Path

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

# Reproducibility
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

# Project directory
PROJECT_DIR = Path.cwd()

# Data directories
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Model configuration
MODEL_NAME = "valhalla/distilbart-mnli-12-1"

# Hardware detection
DEVICE = 0 if torch.cuda.is_available() else -1

if torch.cuda.is_available():

    print("Execution device: GPU")
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

else:

    print("Execution device: CPU")
    print("No CUDA-enabled GPU detected.")

print("\nPython:", sys.version)
print("Operating System:", platform.platform())
print("PyTorch:", torch.__version__)
print("Model:", MODEL_NAME)

print("\nProject directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DATA_DIR)
print("Processed data directory:", PROCESSED_DATA_DIR)

Execution device: GPU
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
CUDA version: 12.8

Python: 3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]
Operating System: Windows-11-10.0.26200-SP0
PyTorch: 2.11.0+cu128
Model: valhalla/distilbart-mnli-12-1

Project directory: c:\Users\shyam\python_test\PayPal_Hackathon\notebooks
Raw data directory: c:\Users\shyam\python_test\PayPal_Hackathon\notebooks\data\raw
Processed data directory: c:\Users\shyam\python_test\PayPal_Hackathon\notebooks\data\processed


In [10]:
# ============================================================
# CELL 2: DEPENDENCY VALIDATION
# ============================================================

import transformers
from transformers import pipeline
from tqdm.auto import tqdm

print("Transformers version:", transformers.__version__)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("PyTorch version:", torch.__version__)

print("\nAll core dependencies imported successfully.")

W1106 22:43:09.337000 12748 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Transformers version: 5.22.1
Pandas version: 3.1.0
NumPy version: 2.5.4
PyTorch version: 2.11.0+cu128

All core dependencies imported successfully.


In [11]:
# ============================================================
# CELL 3: DATASET DISCOVERY AND LOADING
# ============================================================

csv_files = list(RAW_DATA_DIR.glob("*.csv"))

print("Available CSV files:\n")

for i, file in enumerate(csv_files):
    print(f"{i}: {file.name}")

if not csv_files:
    raise FileNotFoundError(
        f"No CSV files found in {RAW_DATA_DIR}. "
        "Place the review dataset inside data/raw/."
    )

# Select the dataset
# If multiple files exist, change this index.
DATASET_INDEX = 0

DATASET_PATH = csv_files[DATASET_INDEX]

print("\nSelected dataset:", DATASET_PATH.name)

df = pd.read_csv(DATASET_PATH, encoding="latin-1")

print("\nDataset loaded successfully.")
print("Shape:", df.shape)

display(df.head())

Available CSV files:



FileNotFoundError: No CSV files found in c:\Users\shyam\python_test\PayPal_Hackathon\notebooks\data\raw. Place the review dataset inside data/raw/.

In [ ]:
# ============================================================
# CELL 4: INITIAL DATA PROFILING
# ============================================================

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\nMissing values:")
display(
    df.isnull()
    .sum()
    .sort_values(ascending=False)
    .to_frame("Missing Values")
)

print("\nDuplicate review IDs:")

if "reviewId" in df.columns:
    print(df["reviewId"].duplicated().sum())

print("\nDuplicate review text:")

if "content" in df.columns:
    print(df["content"].duplicated().sum())

print("\nRating distribution:")

if "score" in df.columns:
    display(
        df["score"].value_counts().sort_index()
    )

Dataset shape: (28848, 8)

Columns:
['reviewId', 'score', 'content', 'at', 'android_or_ios', 'thumbsUpCount', 'reviewCreatedVersion', 'replyContent']

Data types:


,Data Type
reviewId,str
score,int64
content,str
at,str
android_or_ios,str
thumbsUpCount,float64
reviewCreatedVersion,str
replyContent,str



Missing values:


,Missing Values
replyContent,28753
reviewCreatedVersion,6353
thumbsUpCount,1923
content,2
at,0
score,0
reviewId,0
android_or_ios,0



Duplicate review IDs:
0

Duplicate review text:
4401

Rating distribution:


score
1    11087
2     1503
3     1252
4     1635
5    13371
Name: count, dtype: int64

In [24]:
# ============================================================
# CELL 5: SOURCE SCHEMA VALIDATION
# ============================================================

REQUIRED_COLUMNS = [
    "reviewId",
    "score",
    "content",
    "at",
    "android_or_ios",
    "reviewCreatedVersion",
    "thumbsUpCount"
]

missing_columns = [
    col for col in REQUIRED_COLUMNS
    if col not in df.columns
]

if missing_columns:

    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Source schema validation passed.")

NameError: name 'df' is not defined

In [ ]:
# ============================================================
# CELL 6: DATA CLEANING AND PREPARATION
# ============================================================

# Preserve original data
raw_df = df.copy()

# Remove missing and empty reviews
clean_df = raw_df[
    raw_df["content"].notna() &
    raw_df["content"].astype(str).str.strip().ne("")
].copy()

# Remove duplicate review IDs
clean_df = clean_df.drop_duplicates(
    subset=["reviewId"],
    keep="first"
)

# Standardise text
clean_df["review_text"] = (
    clean_df["content"]
    .astype(str)
    .str.strip()
)

# Standardise ratings
clean_df["review_rating"] = pd.to_numeric(
    clean_df["score"],
    errors="coerce"
)

# Parse timestamps
clean_df["review_timestamp"] = pd.to_datetime(
    clean_df["at"],
    errors="coerce",
    utc=True
)

# Select low-rated reviews
issue_df = clean_df[
    clean_df["review_rating"].isin([1, 2, 3, 4])
].copy()

issue_df.reset_index(drop=True, inplace=True)

print("Original records:", len(raw_df))
print("Clean records:", len(clean_df))
print("Low-rated reviews:", len(issue_df))

print("\nFinal analysis dataset:", issue_df.shape)

display(
    issue_df[
        [
            "reviewId",
            "review_rating",
            "review_text",
            "android_or_ios",
            "reviewCreatedVersion"
        ]
    ].head()
)

Original records: 28848
Clean records: 28846
Low-rated reviews: 15477

Final analysis dataset: (15477, 11)


,reviewId,review_rating,review_text,android_or_ios,reviewCreatedVersion
0,a5a628f9-122a-4819-823f-dedb4777f52d,1,Too much Charges ??,android,10.18.0
1,a75b079c-cafb-4ba2-a662-8992d66ec888,1,"Your agents are dumb, trash and such a big LIARS! Also the app and website and everything in this app is a trash. I can't link my bank card!!! It always shows an error message whenever I try! Now, I contacted an AGENT only to be told by pure lies...",android,10.18.0
2,83993c40-362f-4f55-a0c0-6ce36e6f62ef,1,hacked,android,10.18.0
3,faeb8166-6c7c-41b2-92b8-abcc99232b05,1,app not compatible with tablet devices,android,8.107.0
4,c07892f6-b7a5-495d-83bd-6c2094d31244,1,I will give 1 star only why? because when I reset my password cus I forgot it they only give 5seconds to write the password how can u write it so quick when it takes more than 5 secs to wait for the code..,android,10.18.0


In [12]:
# PayPal_Hackathon/
# │
# ├── data/
# │   ├── raw/
# │   │   └── .gitkeep
# │   ├── processed/
# │   │   └── .gitkeep
# │   └── demo_cases/
# │
# ├── notebooks/
# │   └── review_analysis.ipynb
# │
# ├── backend/
# │   ├── main.py
# │   ├── complaint_parser.py
# │   ├── engineering_intelligence.py
# │   └── README.md
# │
# ├── dashboard/
# │   ├── app.py
# │   └── components/
# │
# ├── database/
# │   └── db.py
# │
# ├── requirements.txt
# ├── .gitignore
# └── README.md

In [13]:
# ============================================================
# CELL 7: MULTI-DIMENSIONAL CLASSIFICATION TAXONOMY
# ============================================================

CLASSIFICATION_TAXONOMY = {

    # --------------------------------------------------------
    # 1. BROAD COMPLAINT CATEGORY
    # --------------------------------------------------------

    "complaint_category": {

        "Payment Issues":
            "Problems with sending, receiving, processing or completing payments",

        "Account Restrictions":
            "Account limitations, account suspension, frozen accounts or blocked accounts",

        "Disputes":
            "Transaction disputes, chargebacks, claims and disagreements over payments",

        "Refund Issues":
            "Delayed refunds, missing refunds, refund processing and money not returned",

        "Bank & Card Issues":
            "Problems linking bank accounts, debit cards, credit cards or withdrawing money",

        "Fraud & Security":
            "Unauthorised transactions, scams, hacking, stolen money and security concerns",

        "Login & Verification":
            "Login failures, password problems, OTP issues, identity verification and account access",

        "Customer Support":
            "Poor customer service, unresolved complaints and difficulty contacting support agents",

        "App Performance":
            "Application crashes, bugs, loading problems, technical errors and slow performance",

        "Fees & Charges":
            "Unexpected fees, transaction charges, currency conversion fees and hidden costs"
    },


    # --------------------------------------------------------
    # 2. ISSUE NATURE
    # --------------------------------------------------------

    "issue_nature": {

        "Bug / Technical Defect":
            "A software malfunction, technical defect or feature that does not work as intended",

        "Logical / Functional Issue":
            "Incorrect business logic, calculation, processing outcome or application behaviour",

        "Enhancement Request":
            "A request to introduce new functionality or additional product capabilities",

        "Feature Improvement":
            "A suggestion to improve an existing feature or make it more useful",

        "UX / Usability Issue":
            "Difficulty navigating, understanding or using an existing application feature",

        "Performance Issue":
            "Slow loading, latency, unresponsiveness or poor application performance",

        "Service / Support Issue":
            "Dissatisfaction with customer service, support quality or complaint resolution",

        "Policy / Restriction Issue":
            "Problems related to account limitations, restrictions, verification or platform policies",

        "Fee / Commercial Issue":
            "Dissatisfaction with fees, charges, pricing or commercial terms",

        "Information / Communication Issue":
            "Missing, unclear, incorrect or misleading information or communication",

        "Other Issue":
            "A customer issue that does not fit the defined classifications"
    },


    # --------------------------------------------------------
    # 3. BUSINESS CONTEXT / INDUSTRY
    # --------------------------------------------------------

    "business_context": {

        "Retail / E-commerce":
            "Online retail stores, e-commerce merchants and digital shopping transactions",

        "Physical Retail / Point of Sale":
            "Physical stores, in-person retail transactions and point-of-sale payments",

        "Food & Beverage":
            "Restaurants, cafes, food delivery businesses and beverage merchants",

        "Travel & Hospitality":
            "Hotels, airlines, travel agencies, tourism and hospitality businesses",

        "Digital Goods & Subscriptions":
            "Digital products, software subscriptions, streaming services and recurring digital purchases",

        "Professional & Personal Services":
            "Consultants, freelancers, professional services and individual service providers",

        "Marketplaces":
            "Online marketplaces, multi-vendor platforms and platform-based seller transactions",

        "Gaming & Entertainment":
            "Gaming platforms, entertainment services and digital entertainment purchases",

        "Financial Services":
            "Financial institutions, financial products and financial service providers",

        "Non-profit / Donations":
            "Charities, fundraising platforms and donation-related transactions",

        "General Consumer / P2P":
            "Personal payments, peer-to-peer transfers and ordinary consumer transactions"
    },


    # --------------------------------------------------------
    # 4. STAKEHOLDER
    # --------------------------------------------------------

    "stakeholder": {

        "Merchant / Seller":
            "A merchant, business owner or seller accepting payments through PayPal",

        "Consumer / Buyer":
            "A customer or buyer making payments to a merchant",

        "Payment Sender":
            "An individual sending money to another person or recipient",

        "Payment Recipient":
            "An individual receiving money from another person",

        "Merchant and Consumer":
            "An issue affecting both the merchant or seller and the customer or buyer",

        "Developer / API Integrator":
            "A software developer or business integrating PayPal APIs or payment services",

        "Marketplace Operator":
            "An operator managing a marketplace or multi-vendor commerce platform"
    },


    # --------------------------------------------------------
    # 5. BUSINESS OPERATION
    # --------------------------------------------------------

    "business_operation": {

        "Online Checkout":
            "Online payment checkout, payment buttons and merchant checkout experiences",

        "In-store Payments":
            "Physical point-of-sale payments and in-person payment acceptance",

        "Merchant Onboarding":
            "Merchant registration, account setup and business verification",

        "Payment Acceptance":
            "Accepting customer payments through PayPal payment services",

        "Payment Processing":
            "Authorisation, processing, completion and settlement of payments",

        "Payouts & Settlements":
            "Merchant payouts, balance transfers and settlement of funds",

        "Refund Management":
            "Initiating, processing and tracking refunds",

        "Recurring Billing":
            "Subscriptions, automatic payments and recurring billing",

        "International Payments":
            "Cross-border payments, international transfers and currency conversion",

        "Seller Account Management":
            "Merchant account settings, business profiles and seller account management",

        "Merchant Reporting & Reconciliation":
            "Sales reports, transaction records, financial reporting and reconciliation",

        "Personal Account Management":
            "Personal PayPal account settings, balance management and account administration"
    },


    # --------------------------------------------------------
    # 6. REQUEST INTENT
    # --------------------------------------------------------

    "request_intent": {

        "Report a Problem":
            "The customer is reporting a malfunction, failed transaction or product issue",

        "Request a New Feature":
            "The customer wants PayPal to introduce a new product feature or capability",

        "Request an Improvement":
            "The customer suggests improving an existing feature or experience",

        "Request an Explanation":
            "The customer wants clarification about a transaction, policy, charge or account status",

        "Request Refund or Reversal":
            "The customer requests a refund, reversal or return of money",

        "Request Account Restoration":
            "The customer wants an account restriction removed or account access restored",

        "Request Customer Support":
            "The customer requests assistance, intervention or communication from support",

        "Express Dissatisfaction":
            "The customer expresses dissatisfaction without a clearly identifiable request"
    },


    # --------------------------------------------------------
    # 7. BUSINESS IMPACT
    # --------------------------------------------------------

    "business_impact": {

        "Financial Loss / Funds Inaccessible":
            "Money lost, unavailable, withheld, incorrectly deducted or inaccessible",

        "Merchant Revenue Disruption":
            "A merchant is unable to receive payments, complete sales or generate revenue",

        "Transaction Disruption":
            "A payment, transfer, refund or transaction cannot be completed successfully",

        "Customer Experience Degradation":
            "A poor customer experience, frustration or reduced usability",

        "Operational Inefficiency":
            "Additional manual work, reconciliation effort or disruption to business operations",

        "Account Access Disruption":
            "The customer or merchant cannot access or use their PayPal account",

        "No Clear Business Impact":
            "The review does not describe a specific financial, operational or service impact"
    }
}


# ------------------------------------------------------------
# VALIDATE TAXONOMY
# ------------------------------------------------------------

print("CLASSIFICATION DIMENSIONS")
print("=" * 55)

for dimension, labels in CLASSIFICATION_TAXONOMY.items():

    print(f"\n{dimension.upper()}")
    print("Number of labels:", len(labels))

    for label in labels:
        print("  -", label)


print("\nTotal classification dimensions:",
      len(CLASSIFICATION_TAXONOMY))

print(
    "Total candidate labels:",
    sum(
        len(labels)
        for labels in CLASSIFICATION_TAXONOMY.values()
    )
)

CLASSIFICATION DIMENSIONS

COMPLAINT_CATEGORY
Number of labels: 10
  - Payment Issues
  - Account Restrictions
  - Disputes
  - Refund Issues
  - Bank & Card Issues
  - Fraud & Security
  - Login & Verification
  - Customer Support
  - App Performance
  - Fees & Charges

ISSUE_NATURE
Number of labels: 11
  - Bug / Technical Defect
  - Logical / Functional Issue
  - Enhancement Request
  - Feature Improvement
  - UX / Usability Issue
  - Performance Issue
  - Service / Support Issue
  - Policy / Restriction Issue
  - Fee / Commercial Issue
  - Information / Communication Issue
  - Other Issue

BUSINESS_CONTEXT
Number of labels: 11
  - Retail / E-commerce
  - Physical Retail / Point of Sale
  - Food & Beverage
  - Travel & Hospitality
  - Digital Goods & Subscriptions
  - Professional & Personal Services
  - Marketplaces
  - Gaming & Entertainment
  - Financial Services
  - Non-profit / Donations
  - General Consumer / P2P

STAKEHOLDER
Number of labels: 7
  - Merchant / Seller
  - Consum

In [14]:
# ============================================================
# CELL 8: LOAD ZERO-SHOT CLASSIFICATION MODEL
# ============================================================

from transformers import pipeline

print("Loading classification model...")
print("Model:", MODEL_NAME)

classifier = pipeline(
    task="zero-shot-classification",
    model=MODEL_NAME,
    device=DEVICE
)

print("\nModel loaded successfully!")

if torch.cuda.is_available():
    print("Execution device:", torch.cuda.get_device_name(0))
else:
    print("Execution device: CPU")

Loading classification model...
Model: valhalla/distilbart-mnli-12-1


Loading weights: 100%|██████████| 231/231 [00:00<00:00, 49709.31it/s]



Model loaded successfully!
Execution device: NVIDIA GeForce RTX 4050 Laptop GPU


In [15]:
# ============================================================
# CELL 9: CLASSIFICATION CONFIGURATION
# ============================================================

BATCH_SIZE = 8
MAX_LENGTH = 256

# Initial threshold; will be validated later.
DEFAULT_THRESHOLD = 0.35

# Save progress periodically.
CHECKPOINT_INTERVAL = 250

# Output files
CHECKPOINT_DIR = PROCESSED_DATA_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_FILE = CHECKPOINT_DIR / "classification_checkpoint.csv"

CLASSIFICATION_OUTPUT = (
    PROCESSED_DATA_DIR / "classified_reviews.csv"
)

print("Batch size:", BATCH_SIZE)
print("Maximum tokens:", MAX_LENGTH)
print("Default threshold:", DEFAULT_THRESHOLD)
print("Checkpoint interval:", CHECKPOINT_INTERVAL)
print("Checkpoint directory:", CHECKPOINT_DIR)

Batch size: 8
Maximum tokens: 256
Default threshold: 0.35
Checkpoint interval: 250
Checkpoint directory: c:\Users\shyam\python_test\PayPal_Hackathon\notebooks\data\processed\checkpoints


In [16]:
# ============================================================
# CELL 10: REUSABLE CLASSIFICATION ENGINE
# ============================================================

import gc
import time

UNKNOWN_LABEL = "Unknown / Insufficient information"


def classify_dimension(
    texts,
    dimension,
    batch_size=BATCH_SIZE,
    threshold=DEFAULT_THRESHOLD,
    taxonomy=None
):

    if taxonomy is None:
        taxonomy = CLASSIFICATION_TAXONOMY

    if dimension not in taxonomy:
        raise ValueError(f"Unknown dimension: {dimension}")

    label_descriptions = taxonomy[dimension]

    candidate_labels = list(label_descriptions.values())

    if len(set(candidate_labels)) != len(candidate_labels):
        raise ValueError(
            f"Duplicate descriptions detected in {dimension}"
        )

    description_to_label = {
        description: label
        for label, description in label_descriptions.items()
    }

    results_output = []

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc=f"Classifying {dimension}"
    ):

        batch_texts = [
            str(text) if pd.notna(text) else ""
            for text in texts[start:start + batch_size]
        ]

        results = classifier(
            batch_texts,
            candidate_labels=candidate_labels,
            hypothesis_template="This customer review describes {}.",
            multi_label=True,
            batch_size=batch_size,
            truncation=True,
            max_length=MAX_LENGTH
        )

        # Handle a single-result batch.
        if isinstance(results, dict):
            results = [results]

        for result in results:

            scored_labels = [
                (
                    description_to_label[label],
                    float(score)
                )
                for label, score in zip(
                    result["labels"],
                    result["scores"]
                )
            ]

            scored_labels.sort(
                key=lambda x: x[1],
                reverse=True
            )

            matched = [
                label
                for label, score in scored_labels
                if score >= threshold
            ]

            primary_label = (
                scored_labels[0][0]
                if scored_labels[0][1] >= threshold
                else UNKNOWN_LABEL
            )

            if not matched:
                matched = [UNKNOWN_LABEL]

            results_output.append({

                f"{dimension}_primary": primary_label,

                f"{dimension}_all": json.dumps(
                    matched,
                    ensure_ascii=False
                ),

                f"{dimension}_scores": json.dumps(
                    {
                        label: round(score, 4)
                        for label, score in scored_labels
                    },
                    ensure_ascii=False
                ),

                f"{dimension}_confidence": round(
                    scored_labels[0][1],
                    4
                )

            })

    return pd.DataFrame(results_output)

In [17]:
# ============================================================
# CELL 10.1: PRODUCT FEATURE & ISSUE LOCATION TAXONOMY
# ============================================================

PRODUCT_LOCATION_TAXONOMY = {

    # --------------------------------------------------------
    # 1. PAYPAL PRODUCT MODULE
    # --------------------------------------------------------

    "product_module": {

        "Account & Security":
            "PayPal account access, account security, restrictions and account settings",

        "Wallet & Balance":
            "PayPal wallet, available balance, stored funds and balance management",

        "Payments & Transfers":
            "Sending money, receiving money, transferring funds and payment processing",

        "Bank & Card Management":
            "Linking, adding, removing and managing bank accounts, debit cards and credit cards",

        "Merchant Payments & Checkout":
            "PayPal checkout, online purchases, merchant payment acceptance and payment buttons",

        "Refunds & Returns":
            "Refund initiation, refund tracking, reversals and returned payments",

        "Disputes & Purchase Protection":
            "Dispute filing, claims, chargebacks, buyer protection and seller protection",

        "Subscriptions & Automatic Payments":
            "Recurring payments, subscriptions, automatic billing and payment agreements",

        "Identity & Verification":
            "Identity checks, KYC, OTP, document verification and account confirmation",

        "Customer Support":
            "Contacting PayPal support, customer service and complaint resolution",

        "App Experience":
            "Application navigation, screens, loading, crashes and general usability",

        "Unknown / Not Identifiable":
            "The review does not provide enough information to identify a PayPal product module"
    },


    # --------------------------------------------------------
    # 2. SPECIFIC PRODUCT FEATURE
    # --------------------------------------------------------

    "product_feature": {

        "Login & Authentication":
            "Logging in, password reset, OTP authentication and account sign-in",

        "Account Access & Restrictions":
            "Accessing an account, restoring an account or resolving account limitations",

        "Add or Link Bank Account":
            "Adding, linking, confirming or removing a bank account",

        "Add or Manage Payment Card":
            "Adding, linking, verifying or removing debit and credit cards",

        "Wallet Balance":
            "Viewing, checking or managing the available PayPal balance",

        "Send Money":
            "Sending money to another PayPal user or payment recipient",

        "Receive Money":
            "Receiving money from another user or merchant",

        "Withdraw Money":
            "Transferring PayPal funds to a linked bank account or eligible card",

        "Transaction History":
            "Viewing payment activity, transaction records and payment details",

        "Online Checkout":
            "Paying merchants through PayPal at online checkout",

        "Payment Confirmation":
            "Confirming, authorising or completing a payment",

        "Refund Tracking":
            "Checking refund status, refund progress or returned funds",

        "Dispute Filing":
            "Opening, submitting or managing a transaction dispute or claim",

        "Purchase Protection":
            "Using buyer protection or seller protection for eligible transactions",

        "Recurring Payments":
            "Managing subscriptions, automatic payments and recurring transactions",

        "Currency Conversion":
            "Converting currencies and viewing exchange rates or conversion charges",

        "Identity Verification":
            "Submitting identity details, documents or verification information",

        "Notifications & Alerts":
            "Receiving or managing payment notifications and account alerts",

        "Customer Support Access":
            "Contacting support, using help features or communicating with support agents",

        "Unknown / Not Identifiable":
            "The review does not provide enough information to identify a specific feature"
    },


    # --------------------------------------------------------
    # 3. WORKFLOW STAGE / ISSUE LOCATION
    # --------------------------------------------------------

    "workflow_stage": {

        "Registration / Account Setup":
            "Creating a PayPal account or completing initial account setup",

        "Login / Account Access":
            "Signing in, authenticating or accessing an existing account",

        "Identity Verification":
            "Completing identity checks, OTP checks or document verification",

        "Bank / Card Linking":
            "Adding, linking or confirming a bank account or payment card",

        "Payment Initiation":
            "Starting a payment or entering payment details",

        "Payment Authorisation":
            "Approving, authenticating or authorising a payment",

        "Payment Processing":
            "Processing or executing a payment after it has been initiated",

        "Payment Confirmation":
            "Receiving confirmation or the final status of a payment",

        "Fund Receipt":
            "Receiving money into a PayPal account or wallet",

        "Withdrawal / Payout":
            "Moving available PayPal funds to a bank account or card",

        "Refund Processing":
            "Initiating or waiting for a refund or reversal",

        "Dispute Resolution":
            "Submitting evidence, reviewing or resolving a dispute or claim",

        "Subscription Management":
            "Starting, changing or cancelling recurring payments",

        "Customer Support Interaction":
            "Seeking assistance or following up on a complaint",

        "General App Navigation":
            "Navigating the application without a clearly identifiable transaction stage",

        "Unknown / Not Identifiable":
            "The review does not provide enough information to identify a workflow stage"
    },


    # --------------------------------------------------------
    # 4. REPORTED FEATURE SYMPTOM
    # --------------------------------------------------------

    "reported_symptom": {

        "Feature Not Working":
            "A feature or function does not work when the customer attempts to use it",

        "Transaction Failed / Declined":
            "A payment or transfer fails, is declined or cannot be completed",

        "Unexpected Transaction Outcome":
            "A transaction produces an unexpected or incorrect result",

        "Delayed Processing":
            "A payment, transfer, refund or other process takes longer than expected",

        "Funds Missing / Not Received":
            "Expected funds have not appeared in the account or recipient balance",

        "Funds Held / Inaccessible":
            "Money is held, restricted, unavailable or cannot be accessed",

        "Verification Failure":
            "Identity, OTP, bank or card verification fails",

        "App Crash / Technical Error":
            "The application crashes, displays errors or stops functioning",

        "Slow / Unresponsive Interface":
            "The application or a feature loads slowly or does not respond",

        "Incorrect Information Displayed":
            "The application displays incorrect, inconsistent or misleading information",

        "Navigation / Usability Difficulty":
            "The customer cannot easily find, understand or use a feature",

        "Duplicate Charge / Transaction":
            "A customer reports being charged or processed more than once",

        "Unauthorised Activity":
            "The customer reports an unrecognised or unauthorised transaction or account activity",

        "Missing Feature / Capability":
            "The customer reports that a desired capability is unavailable",

        "Poor Support / No Resolution":
            "The customer reports inadequate support or an unresolved issue",

        "Unknown / Not Identifiable":
            "The review does not provide enough information to identify a specific symptom"
    }

}


# ------------------------------------------------------------
# VALIDATE NEW TAXONOMY
# ------------------------------------------------------------

print("PRODUCT FEATURE & ISSUE LOCATION DIMENSIONS")
print("=" * 60)

for dimension, labels in PRODUCT_LOCATION_TAXONOMY.items():

    print(f"\n{dimension.upper()}")
    print("Number of labels:", len(labels))

    for label in labels:
        print("  -", label)


print("\nNew dimensions:", len(PRODUCT_LOCATION_TAXONOMY))

print(
    "New candidate labels:",
    sum(len(labels) for labels in PRODUCT_LOCATION_TAXONOMY.values())
)

PRODUCT FEATURE & ISSUE LOCATION DIMENSIONS

PRODUCT_MODULE
Number of labels: 12
  - Account & Security
  - Wallet & Balance
  - Payments & Transfers
  - Bank & Card Management
  - Merchant Payments & Checkout
  - Refunds & Returns
  - Disputes & Purchase Protection
  - Subscriptions & Automatic Payments
  - Identity & Verification
  - Customer Support
  - App Experience
  - Unknown / Not Identifiable

PRODUCT_FEATURE
Number of labels: 20
  - Login & Authentication
  - Account Access & Restrictions
  - Add or Link Bank Account
  - Add or Manage Payment Card
  - Wallet Balance
  - Send Money
  - Receive Money
  - Withdraw Money
  - Transaction History
  - Online Checkout
  - Payment Confirmation
  - Refund Tracking
  - Dispute Filing
  - Purchase Protection
  - Recurring Payments
  - Currency Conversion
  - Identity Verification
  - Notifications & Alerts
  - Customer Support Access
  - Unknown / Not Identifiable

WORKFLOW_STAGE
Number of labels: 16
  - Registration / Account Setup
  - 

In [18]:
# ============================================================
# CELL 10A: COMBINE CLASSIFICATION TAXONOMIES
# ============================================================

ALL_CLASSIFICATION_TAXONOMIES = {
    **CLASSIFICATION_TAXONOMY,
    **PRODUCT_LOCATION_TAXONOMY
}

print("Total dimensions:",
      len(ALL_CLASSIFICATION_TAXONOMIES))

for dimension, labels in ALL_CLASSIFICATION_TAXONOMIES.items():
    print(f"{dimension}: {len(labels)} labels")

Total dimensions: 11
complaint_category: 10 labels
issue_nature: 11 labels
business_context: 11 labels
stakeholder: 7 labels
business_operation: 12 labels
request_intent: 8 labels
business_impact: 7 labels
product_module: 12 labels
product_feature: 20 labels
workflow_stage: 16 labels
reported_symptom: 16 labels


In [19]:
# ============================================================
# CELL 11: SMALL-SCALE CLASSIFICATION TEST
# ============================================================

sample_df = issue_df.head(10).copy()

sample_texts = (
    sample_df["review_text"]
    .astype(str)
    .tolist()
)

TEST_DIMENSIONS = [
    "issue_nature",
    "product_module",
    "product_feature",
    "workflow_stage",
    "reported_symptom"
]

sample_results = []

for dimension in TEST_DIMENSIONS:

    print(f"\nTesting: {dimension}")

    result = classify_dimension(
    texts=sample_texts,
    dimension=dimension,
    taxonomy=ALL_CLASSIFICATION_TAXONOMIES
    )
    
    sample_results.append(result)

sample_output = pd.concat(
    [
        sample_df[
            ["review_rating", "review_text"]
        ].reset_index(drop=True)
    ] + sample_results,
    axis=1
)

display(sample_output)

NameError: name 'issue_df' is not defined

In [20]:
# ============================================================
# CELL 12: CHECKPOINT-ENABLED CLASSIFICATION
# ============================================================


ALL_CLASSIFICATION_TAXONOMIES = {
    **CLASSIFICATION_TAXONOMY,
    **PRODUCT_LOCATION_TAXONOMY
}

DIMENSIONS_TO_CLASSIFY = list(
    ALL_CLASSIFICATION_TAXONOMIES.keys()
)

print("Total dimensions:", len(DIMENSIONS_TO_CLASSIFY))

for dimension in DIMENSIONS_TO_CLASSIFY:
    print(
        dimension,
        "->",
        len(ALL_CLASSIFICATION_TAXONOMIES[dimension]),
        "labels"
    )
# Create stable internal review keys.
issue_df = issue_df.copy()

issue_df["classification_id"] = issue_df.apply(
    lambda row: hashlib.sha256(
        (
            str(row["reviewId"]) +
            "|" +
            str(row["review_text"])
        ).encode("utf-8")
    ).hexdigest()[:20],
    axis=1
)

# Load existing checkpoint if available.
if CHECKPOINT_FILE.exists():

    checkpoint_df = pd.read_csv(
        CHECKPOINT_FILE,
        dtype={"classification_id": str}
    )

    print(
        "Existing checkpoint loaded:",
        len(checkpoint_df)
    )

else:

    checkpoint_df = pd.DataFrame(
        columns=["classification_id"]
    )

    print("No previous checkpoint found.")

checkpoint_df["classification_id"] = (
    checkpoint_df["classification_id"].astype(str)
)

checkpoint_ids = set(
    checkpoint_df["classification_id"]
)

# Only process records not already completed.
pending_df = issue_df[
    ~issue_df["classification_id"].isin(checkpoint_ids)
].copy()

print("\nTotal reviews:", len(issue_df))
print("Already classified:", len(checkpoint_df))
print("Remaining:", len(pending_df))

Total dimensions: 11
complaint_category -> 10 labels
issue_nature -> 11 labels
business_context -> 11 labels
stakeholder -> 7 labels
business_operation -> 12 labels
request_intent -> 8 labels
business_impact -> 7 labels
product_module -> 12 labels
product_feature -> 20 labels
workflow_stage -> 16 labels
reported_symptom -> 16 labels


NameError: name 'issue_df' is not defined

In [21]:

# ============================================================
# CELL 12A: RESUMABLE CLASSIFICATION WITH BATCH FILES
# ============================================================

BATCH_CHECKPOINT_DIR = CHECKPOINT_DIR / "batches"
BATCH_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def run_resumable_classification(
    data,
    dimensions,
    batch_size=BATCH_SIZE
):

    data = data.copy()
    data["classification_id"] = (
        data["classification_id"].astype(str)
    )

    for dimension in dimensions:

        print("\n" + "=" * 60)
        print("PROCESSING:", dimension)
        print("=" * 60)

        # Existing consolidated checkpoint
        dimension_file = (
            CHECKPOINT_DIR / f"classification_{dimension}.csv"
        )

        # New folder containing individual batch files
        batch_dir = BATCH_CHECKPOINT_DIR / dimension
        batch_dir.mkdir(parents=True, exist_ok=True)

        completed_frames = []

        # 1. Load existing checkpoint, if available
        if dimension_file.exists():

            existing_df = pd.read_csv(
                dimension_file,
                dtype={"classification_id": str}
            )

            completed_frames.append(existing_df)

            print("Existing checkpoint records:", len(existing_df))

        # 2. Load previously saved batch files
        batch_files = sorted(batch_dir.glob("batch_*.csv"))

        for batch_file in batch_files:

            try:
                saved_batch = pd.read_csv(
                    batch_file,
                    dtype={"classification_id": str}
                )

                if not saved_batch.empty:
                    completed_frames.append(saved_batch)

            except Exception as e:
                print(f"Could not read {batch_file.name}: {e}")
                raise

        # 3. Combine saved records and identify completed IDs
        if completed_frames:

            completed_df = pd.concat(
                completed_frames,
                ignore_index=True
            )

            completed_df = completed_df.drop_duplicates(
                subset=["classification_id"],
                keep="last"
            )

            completed_ids = set(
                completed_df["classification_id"].dropna()
            )

        else:

            completed_df = pd.DataFrame()
            completed_ids = set()

        # 4. Identify pending reviews
        pending = data[
            ~data["classification_id"].isin(completed_ids)
        ].copy()

        print("Total reviews:", len(data))
        print("Completed:", len(completed_ids))
        print("Remaining:", len(pending))

        if pending.empty:
            print("Dimension already completed.")
            continue

        # 5. Process and save individual batches
        for start in tqdm(
            range(0, len(pending), batch_size),
            desc=dimension
        ):

            batch = pending.iloc[
                start:start + batch_size
            ].copy()

            batch_results = classify_dimension(
                texts=batch["review_text"].astype(str).tolist(),
                dimension=dimension,
                batch_size=batch_size,
                taxonomy=ALL_CLASSIFICATION_TAXONOMIES
            )

            batch_results = batch_results.reset_index(drop=True)

            batch_results.insert(
                0,
                "classification_id",
                batch["classification_id"].tolist()
            )

            # Unique filename based on the first review ID
            batch_filename = (
                "batch_" +
                batch["classification_id"].iloc[0] +
                ".csv"
            )

            batch_filepath = batch_dir / batch_filename

            # Save this batch directly; never replace an existing CSV
            if batch_filepath.exists():
                raise FileExistsError(
                    f"Batch checkpoint already exists: {batch_filepath}"
                )

            batch_results.to_csv(
                batch_filepath,
                index=False
            )

        # 6. Consolidate into a separate, versioned output file
        all_batch_frames = [completed_df] if not completed_df.empty else []

        for batch_file in sorted(batch_dir.glob("batch_*.csv")):

            saved_batch = pd.read_csv(
                batch_file,
                dtype={"classification_id": str}
            )

            all_batch_frames.append(saved_batch)

        final_df = pd.concat(
            all_batch_frames,
            ignore_index=True
        )

        final_df = final_df.drop_duplicates(
            subset=["classification_id"],
            keep="last"
        )

        # Preserve original review order
        order_map = {
            review_id: index
            for index, review_id in enumerate(data["classification_id"])
        }

        final_df["_order"] = final_df["classification_id"].map(order_map)
        final_df = (
            final_df.sort_values("_order")
            .drop(columns="_order")
            .reset_index(drop=True)
        )

        # New filename: original checkpoint remains untouched
        timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

        output_file = (
            CHECKPOINT_DIR /
            f"classification_{dimension}_complete_{timestamp}.csv"
        )

        final_df.to_csv(output_file, index=False)

        print("\nCompleted dimension:", dimension)
        print("Unique classified reviews:", len(final_df))
        print("Saved consolidated file:", output_file)

        del completed_df, final_df, pending
        gc.collect()

    print("\nAll requested dimensions processed.")


In [22]:
# ============================================================
# CHECK EXISTING CLASSIFICATION PROGRESS
# ============================================================

dimension_file = (
    CHECKPOINT_DIR /
    "classification_complaint_category.csv"
)

if dimension_file.exists():

    saved_df = pd.read_csv(dimension_file)

    print("Saved records:", len(saved_df))

    print(
        "Unique review IDs:",
        saved_df["classification_id"].nunique()
    )

    print(
        "Total reviews:",
        len(issue_df)
    )

    print(
        "Remaining reviews:",
        len(issue_df) -
        saved_df["classification_id"].nunique()
    )

else:

    print("No completed checkpoint file found.")

No completed checkpoint file found.


In [23]:
# run_resumable_classification(
#     data=issue_df,
#     dimensions=DIMENSIONS_TO_CLASSIFY
# )


run_resumable_classification(
    data=issue_df,
    dimensions=["issue_nature"],
    batch_size=BATCH_SIZE
)


NameError: name 'issue_df' is not defined

In [90]:
# ============================================================
# DIAGNOSE EXISTING CHECKPOINT FILES
# ============================================================

for dimension in ["complaint_category", "issue_nature"]:

    filepath = (
        CHECKPOINT_DIR /
        f"classification_{dimension}.csv"
    )

    temp_filepath = (
        CHECKPOINT_DIR /
        f"classification_{dimension}_temp.csv"
    )

    print(f"\n{dimension.upper()}")
    print("-" * 45)

    print("Checkpoint exists:", filepath.exists())
    print("Temporary file exists:", temp_filepath.exists())

    if filepath.exists():

        try:
            saved = pd.read_csv(filepath)

            print("Saved records:", len(saved))
            print(
                "Unique IDs:",
                saved["classification_id"].nunique()
            )

        except Exception as e:
            print("Could not read checkpoint:", repr(e))


COMPLAINT_CATEGORY
---------------------------------------------
Checkpoint exists: True
Temporary file exists: False
Saved records: 15477
Unique IDs: 15477

ISSUE_NATURE
---------------------------------------------
Checkpoint exists: True
Temporary file exists: False
Saved records: 15477
Unique IDs: 15477


In [2]:

# ============================================================
# CELL 13: VALIDATE COMPLETED CLASSIFICATIONS
# ============================================================

dimensions_to_validate = [
    "complaint_category",
    "issue_nature"
]

source_ids = set(
    issue_df["classification_id"].astype(str)
)

for dimension in dimensions_to_validate:

    print("\n" + "=" * 65)
    print("VALIDATING:", dimension.upper())
    print("=" * 65)

    filepath = (
        CHECKPOINT_DIR /
        f"classification_{dimension}.csv"
    )

    if not filepath.exists():
        print("ERROR: File not found:", filepath)
        continue

    result_df = pd.read_csv(
        filepath,
        dtype={"classification_id": str}
    )

    expected_columns = [
        "classification_id",
        f"{dimension}_primary",
        f"{dimension}_all",
        f"{dimension}_scores",
        f"{dimension}_confidence"
    ]

    missing_columns = [
        col for col in expected_columns
        if col not in result_df.columns
    ]

    result_ids = set(
        result_df["classification_id"].dropna().astype(str)
    )

    print("Total records:", len(result_df))
    print("Unique IDs:", result_df["classification_id"].nunique())
    print("Duplicate IDs:", result_df["classification_id"].duplicated().sum())

    print("Missing columns:", missing_columns)

    print("Missing primary labels:",
          result_df[f"{dimension}_primary"].isna().sum())

    print("Missing confidence scores:",
          result_df[f"{dimension}_confidence"].isna().sum())

    print("IDs missing from results:",
          len(source_ids - result_ids))

    print("Unexpected IDs:",
          len(result_ids - source_ids))

    print("\nPrimary label distribution:")

    print(
        result_df[f"{dimension}_primary"]
        .value_counts(dropna=False)
        .to_string()
    )

    if (
        len(result_df) == len(issue_df)
        and result_df["classification_id"].nunique() == len(issue_df)
        and not missing_columns
        and result_ids == source_ids
    ):
        print("\nRESULT: STRUCTURAL VALIDATION PASSED")

    else:
        print("\nRESULT: VALIDATION NEEDS ATTENTION")


NameError: name 'issue_df' is not defined

In [5]:
# ============================================================
# STAGE 1: EXPORT CLASSIFIED REVIEWS FOR PAYPAL SENTINEL
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1. Resolve paths
# ------------------------------------------------------------

PROJECT_DIR = Path.cwd()

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
CHECKPOINT_DIR = PROCESSED_DIR / "checkpoints"

COMPLAINT_FILE = CHECKPOINT_DIR / "classification_complaint_category.csv"
ISSUE_NATURE_FILE = CHECKPOINT_DIR / "classification_issue_nature.csv"

OUTPUT_FILE = PROCESSED_DIR / "classified_reviews_stage1.csv"

# ------------------------------------------------------------
# 2. Check that the required data exists
# ------------------------------------------------------------

if "issue_df" not in globals():
    raise NameError(
        "issue_df is not available. Run your data preparation "
        "and classification notebook cells first."
    )

if not COMPLAINT_FILE.exists():
    raise FileNotFoundError(COMPLAINT_FILE)

if not ISSUE_NATURE_FILE.exists():
    raise FileNotFoundError(ISSUE_NATURE_FILE)

if "classification_id" not in issue_df.columns:
    raise KeyError("classification_id is missing from issue_df.")

# ------------------------------------------------------------
# 3. Load existing classification checkpoints
# ------------------------------------------------------------

complaint_df = pd.read_csv(COMPLAINT_FILE)
issue_nature_df = pd.read_csv(ISSUE_NATURE_FILE)

print("Original reviews:", issue_df.shape)
print("Complaint classifications:", complaint_df.shape)
print("Issue nature classifications:", issue_nature_df.shape)

# ------------------------------------------------------------
# 4. Validate IDs before merging
# ------------------------------------------------------------

for name, frame in {
    "issue_df": issue_df,
    "complaint_df": complaint_df,
    "issue_nature_df": issue_nature_df
}.items():

    if frame["classification_id"].isna().any():
        raise ValueError(f"{name} contains missing classification IDs.")

    if frame["classification_id"].duplicated().any():
        raise ValueError(f"{name} contains duplicate classification IDs.")

# ------------------------------------------------------------
# 5. Merge classifications with the original reviews
# ------------------------------------------------------------

sentinel_df = issue_df.copy()

sentinel_df = sentinel_df.merge(
    complaint_df,
    on="classification_id",
    how="left",
    validate="one_to_one"
)

sentinel_df = sentinel_df.merge(
    issue_nature_df,
    on="classification_id",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 6. Validate the consolidated dataset
# ------------------------------------------------------------

assert len(sentinel_df) == len(issue_df), (
    "Review count changed during merging."
)

assert sentinel_df["classification_id"].is_unique, (
    "Duplicate IDs found in consolidated dataset."
)

required_columns = [
    "classification_id",
    "review_text",
    "complaint_category_primary",
    "issue_nature_primary"
]

missing_columns = [
    col for col in required_columns
    if col not in sentinel_df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns missing: {missing_columns}"
    )

print("\nMissing complaint classifications:",
      sentinel_df["complaint_category_primary"].isna().sum())

print("Missing issue nature classifications:",
      sentinel_df["issue_nature_primary"].isna().sum())

# ------------------------------------------------------------
# 7. Save as a NEW application dataset
# ------------------------------------------------------------

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

sentinel_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8"
)

print("\nSTAGE 1 EXPORT COMPLETE")
print("Output:", OUTPUT_FILE)
print("Final shape:", sentinel_df.shape)
print("Unique reviews:", sentinel_df["classification_id"].nunique())
print("\nColumns:")
print(sentinel_df.columns.tolist())

NameError: issue_df is not available. Run your data preparation and classification notebook cells first.

In [3]:
# ============================================================
# RECOVERY STEP 1: LOCATE PROJECT FILES
# ============================================================

from pathlib import Path
import pandas as pd

# Find project root
PROJECT_DIR = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "data").exists()
    ),
    None
)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not locate the project directory. "
        "Please navigate to the PayPal_Hackathon folder."
    )

print("PROJECT DIRECTORY:")
print(PROJECT_DIR)

# Define paths
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
CHECKPOINT_DIR = PROCESSED_DIR / "checkpoints"

print("\nCSV FILES FOUND:")
for file in PROJECT_DIR.rglob("*.csv"):
    print(file.relative_to(PROJECT_DIR))

print("\nCHECKPOINT FILES:")
for file in CHECKPOINT_DIR.glob("*.csv"):
    print(file.name)

PROJECT DIRECTORY:
c:\Users\shyam\python_test\PayPal_Hackathon

CSV FILES FOUND:
venv\Lib\site-packages\tornado\test\csv_translations\fr_FR.csv
venv\Lib\site-packages\sklearn\datasets\data\breast_cancer.csv
venv\Lib\site-packages\sklearn\datasets\data\iris.csv
venv\Lib\site-packages\sklearn\datasets\data\linnerud_exercise.csv
venv\Lib\site-packages\sklearn\datasets\data\linnerud_physiological.csv
venv\Lib\site-packages\sklearn\datasets\data\wine_data.csv
venv\Lib\site-packages\scipy\signal\tests\data\GLB.Ts+dSST.csv
venv\Lib\site-packages\numpy\_core\tests\data\umath-validation-set-arccos.csv
venv\Lib\site-packages\numpy\_core\tests\data\umath-validation-set-arccosh.csv
venv\Lib\site-packages\numpy\_core\tests\data\umath-validation-set-arcsin.csv
venv\Lib\site-packages\numpy\_core\tests\data\umath-validation-set-arcsinh.csv
venv\Lib\site-packages\numpy\_core\tests\data\umath-validation-set-arctan.csv
venv\Lib\site-packages\numpy\_core\tests\data\umath-validation-set-arctanh.csv
venv\Li

In [4]:
# ============================================================
# RECOVERY STEP 2: INSPECT CHECKPOINTS
# ============================================================

COMPLAINT_FILE = (
    CHECKPOINT_DIR / "classification_complaint_category.csv"
)

ISSUE_NATURE_FILE = (
    CHECKPOINT_DIR / "classification_issue_nature.csv"
)

complaint_df = pd.read_csv(COMPLAINT_FILE)
issue_nature_df = pd.read_csv(ISSUE_NATURE_FILE)

print("COMPLAINT CLASSIFICATION")
print("Shape:", complaint_df.shape)
print("Columns:", complaint_df.columns.tolist())
display(complaint_df.head(3))

print("\nISSUE NATURE CLASSIFICATION")
print("Shape:", issue_nature_df.shape)
print("Columns:", issue_nature_df.columns.tolist())
display(issue_nature_df.head(3))

COMPLAINT CLASSIFICATION
Shape: (15477, 5)
Columns: ['classification_id', 'complaint_category_primary', 'complaint_category_all', 'complaint_category_scores', 'complaint_category_confidence']


,classification_id,complaint_category_primary,complaint_category_all,complaint_category_scores,complaint_category_confidence
0,88f48b366dd4f475d73d,Fees & Charges,"[""Fees & Charges""]","{""Fees & Charges"": 0.8943, ""Bank & Card Issues...",0.8943
1,d080265c91520f2a61c4,Bank & Card Issues,"[""Bank & Card Issues"", ""Customer Support"", ""Pa...","{""Bank & Card Issues"": 0.9686, ""Customer Suppo...",0.9686
2,0a58b014ce0b96c0d258,Fraud & Security,"[""Fraud & Security""]","{""Fraud & Security"": 0.7084, ""Fees & Charges"":...",0.7084



ISSUE NATURE CLASSIFICATION
Shape: (15477, 5)
Columns: ['classification_id', 'issue_nature_primary', 'issue_nature_all', 'issue_nature_scores', 'issue_nature_confidence']


,classification_id,issue_nature_primary,issue_nature_all,issue_nature_scores,issue_nature_confidence
0,88f48b366dd4f475d73d,Fee / Commercial Issue,"[""Fee / Commercial Issue""]","{""Fee / Commercial Issue"": 0.9042, ""Service / ...",0.9042
1,d080265c91520f2a61c4,Bug / Technical Defect,"[""Bug / Technical Defect"", ""Service / Support ...","{""Bug / Technical Defect"": 0.9825, ""Service / ...",0.9825
2,0a58b014ce0b96c0d258,Bug / Technical Defect,"[""Bug / Technical Defect""]","{""Bug / Technical Defect"": 0.7812, ""Service / ...",0.7812


In [8]:
# ============================================================
# RECOVERY STEP 3: FIND RAW REVIEW DATA
# ============================================================

csv_files = list(RAW_DIR.rglob("*.csv"))

print("RAW CSV FILES FOUND:", len(csv_files))

for i, file in enumerate(csv_files):
    print(f"\n[{i+1}] {file.name}")

    temp_df = pd.read_csv(file, nrows=3)

    print("Columns:", temp_df.columns.tolist())
    display(temp_df.head(2))

RAW CSV FILES FOUND: 1

[1] Paypal_Android_iOS_Reviews_2026.csv
Columns: ['reviewId', 'score', 'content', 'at', 'android_or_ios', 'thumbsUpCount', 'reviewCreatedVersion', 'replyContent']


,reviewId,score,content,at,android_or_ios,thumbsUpCount,reviewCreatedVersion,replyContent
0,8938a0da-4273-4019-865c-16ebf97e251a,5,brilliant,10/26/2026 15:18,android,0,10.14.0,NaN
1,a5a628f9-122a-4819-823f-dedb4777f52d,1,Too much Charges ??,10/26/2026 14:55,android,0,10.18.0,NaN
